### description + vendor_name

load,clean,split

In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1) Load dataset
df = pd.read_csv("datasets/dataset1_text_rich_transactions_harder_v2.csv")  # your attached file
# transaction_date is dd-mm-YYYY
df["transaction_date"] = pd.to_datetime(df["transaction_date"], format="%d-%m-%Y", errors="coerce")

# 2) Basic filtering: drop rows with missing critical fields
df = df.dropna(subset=["description", "category_label", "transaction_date"])

# 3) Text cleaning: robust regex-based normalization
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    # remove common bank noise tokens you don't want to dominate
    noise_tokens = [
        r"\bfy\d{2}\b",          # fy24, fy25
        r"\bq[1-4]\b",           # q1, q2
        r"\binv/?\d*\b",         # inv, inv/2404/001
        r"\bbill/?\d*\b",        # bill/...
        r"\brcpt/?\d*\b",        # rcpt/...
        r"\btaxinv/?\d*\b",      # taxinv/...
        r"\bref\b\s*\d+",        # ref 1234
        r"\badv\b",              # adv
        r"\bsubs\b",             # subs
        r"\bgst\b",              # gst word (optional)
    ]
    for pat in noise_tokens:
        s = re.sub(pat, " ", s)
    # keep alphanumerics and basic separators
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["description_clean"] = df["description"].apply(clean_text)

# Optional: also normalize vendor_name for some models
def clean_vendor(v):
    if not isinstance(v, str):
        return ""
    v = v.upper()
    v = re.sub(r"\bPVT\.?\b|\bLTD\.?\b|\bLIMITED\b|\bINDIA\b", " ", v)
    v = re.sub(r"[^A-Z0-9\s]", " ", v)
    v = re.sub(r"\s+", " ", v).strip()
    return v

df["vendor_clean"] = df["vendor_name"].apply(clean_vendor)

# 4) Simple date features
df["month"] = df["transaction_date"].dt.month
df["dow"] = df["transaction_date"].dt.dayofweek  # 0=Monday

# 5) Define features/target
TEXT_COL = "description_clean"
VENDOR_COL = "vendor_clean"
NUM_COLS = ["amount", "month", "dow"]
TARGET_COL = "category_label"

X_text = df[[TEXT_COL, VENDOR_COL] + NUM_COLS]
y = df[TARGET_COL]

# 6) Train-test split (stratified to respect imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, stratify=y, random_state=42
)


In [3]:
df.head()

,transaction_id,transaction_date,amount,currency,description,vendor_name,gst_applicable,gst_slab,itc_eligible,category_label,is_anomaly,description_clean,vendor_clean,month,dow
0,TXN0000001,2024-04-01,7641.92,INR,service fees,IKEA,True,exempt,false,Office Supplies,0,service fees,IKEA,4,0
1,TXN0000002,2024-04-01,58560.14,INR,upi awfis rcpt/2404/002 fy24 - ent ex# ref 6160,Awfis,True,12%,unknown,Rent,0,upi awfis 002 ent ex,AWFIS,4,0
2,TXN0000003,2024-04-01,2184.95,INR,bill paid,ZOMATO,True,5%,FALSE,Office Supplies,0,paid,ZOMATO,4,0
3,TXN0000004,2024-04-01,5375.79,INR,!UPI UBER INDIA PVT LTD TAXINV/2501/924 2 - ex...,OFFICE DEPOT Pvt Ltd,True,exempt,unknown,Office Supplies,0,upi uber india pvt ltd 924 2 exp rf 7593,OFFICE DEPOT,4,0
4,TXN0000005,2024-04-01,68830.10,INR,auto debit,AWFIS INDIA PVT LTD,True,18%,TRUE,Rent,0,auto debit,AWFIS,4,0


TF‑IDF + Logistic Regression


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

# 1) Build combined text feature: description + vendor
for df_part in (X_train, X_test):
    df_part["text_combo"] = (
        df_part[TEXT_COL].fillna("") + " " + df_part[VENDOR_COL].fillna("")
    )

TEXT_COMBO = "text_combo"

# 2) Preprocessor: TF-IDF on text, scaling on numeric
tfidf = TfidfVectorizer(
    max_features=8000,          # high enough for variety, small enough for speed
    ngram_range=(1, 2),         # unigrams + bigrams
    min_df=3,                   # drop very rare terms
)

preprocess_lr = ColumnTransformer(
    transformers=[
        ("text", tfidf, TEXT_COMBO),
        ("num", StandardScaler(), NUM_COLS),
    ],
    remainder="drop",
)

# 3) Model: Logistic Regression (multinomial)
log_reg = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    multi_class="multinomial",
    class_weight="balanced"     # helpful for class imbalance
)

pipe_lr = Pipeline(
    steps=[
        ("preprocess", preprocess_lr),
        ("clf", log_reg),
    ]
)

# 4) Train
pipe_lr.fit(X_train, y_train)

# 5) Evaluate
y_pred_lr = pipe_lr.predict(X_test)
print("=== TF-IDF + Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))


c:\ProgramData\anaconda3\envs\exassaro\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


=== TF-IDF + Logistic Regression ===
                 precision    recall  f1-score   support

Exempt Services       0.96      1.00      0.98        26
    IT Services       0.97      0.81      0.89       209
          Meals       0.74      0.93      0.82        55
Office Supplies       0.98      0.93      0.96       128
           Rent       0.99      0.99      0.99       151
       Software       0.99      0.99      0.99       115
       Training       0.73      0.88      0.80        76
         Travel       0.94      0.94      0.94        90
      Utilities       0.88      0.95      0.91       125

       accuracy                           0.92       975
      macro avg       0.91      0.94      0.92       975
   weighted avg       0.93      0.92      0.92       975



TF‑IDF + XGBoost (or LightGBM)


In [5]:
from sklearn.preprocessing import LabelEncoder

# y is currently category_label as strings
le = LabelEncoder()
y_encoded = le.fit_transform(y)          # maps classes to 0..n-1

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)


In [6]:
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# reuse TEXT_COMBO and NUM_COLS as before
for df_part in (X_train, X_test):
    df_part["text_combo"] = (
        df_part["description_clean"].fillna("") + " " + df_part["vendor_clean"].fillna("")
    )

tfidf_xgb = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

preprocess_xgb = ColumnTransformer(
    transformers=[
        ("text", tfidf_xgb, "text_combo"),
        ("num", "passthrough", ["amount", "month", "dow"]),
    ],
    remainder="drop",
)

xgb_clf = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
)

pipe_xgb = Pipeline(
    steps=[
        ("preprocess", preprocess_xgb),
        ("clf", xgb_clf),
    ]
)

pipe_xgb.fit(X_train, y_train)
y_pred_enc = pipe_xgb.predict(X_test)

# Decode back to original string labels for reporting
y_test_str = le.inverse_transform(y_test)
y_pred_str = le.inverse_transform(y_pred_enc)

print("=== TF-IDF + XGBoost (encoded labels) ===")
print(classification_report(y_test_str, y_pred_str))


=== TF-IDF + XGBoost (encoded labels) ===
                 precision    recall  f1-score   support

Exempt Services       0.96      1.00      0.98        26
    IT Services       0.89      0.89      0.89       209
          Meals       0.81      0.78      0.80        55
Office Supplies       0.97      0.92      0.94       128
           Rent       0.99      0.99      0.99       151
       Software       0.99      0.99      0.99       115
       Training       0.72      0.76      0.74        76
         Travel       0.94      0.97      0.95        90
      Utilities       0.93      0.93      0.93       125

       accuracy                           0.92       975
      macro avg       0.91      0.92      0.91       975
   weighted avg       0.92      0.92      0.92       975



LightGBM

In [7]:
from lightgbm import LGBMClassifier

lgbm_clf = LGBMClassifier(
    objective="multiclass",
    num_leaves=63,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
)

pipe_lgbm = Pipeline(
    steps=[
        ("preprocess", preprocess_xgb),  # reuse TF-IDF + num
        ("clf", lgbm_clf),
    ]
)

pipe_lgbm.fit(X_train, y_train)
y_pred_lgbm = pipe_lgbm.predict(X_test)
print("=== TF-IDF + LightGBM ===")
print(classification_report(y_test, y_pred_lgbm))


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001572 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5788
[LightGBM] [Info] Number of data points in the train set: 3900, number of used features: 172
[LightGBM] [Info] Start training from score -3.605293
[LightGBM] [Info] Start training from score -1.541300
[LightGBM] [Info] Start training from score -2.875104
[LightGBM] [Info] Start training from score -2.032362
[LightGBM] [Info] Start training from score -1.863503
[LightGBM] [Info] Start training from score -2.144048
[LightGBM] [Info] Start training from score -2.554999
[LightGBM] [Info] Start training from score -2.377088
[LightGBM] [Info] Start training from score -2.052126
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


c:\ProgramData\anaconda3\envs\exassaro\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Multinomial Naive Bayes (text-only baseline)

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# 1) Use only text (description + vendor) for this baseline
X_train_nb = X_train.copy()
X_test_nb = X_test.copy()
X_train_nb["text_combo"] = (
    X_train_nb[TEXT_COL].fillna("") + " " + X_train_nb[VENDOR_COL].fillna("")
)
X_test_nb["text_combo"] = (
    X_test_nb[TEXT_COL].fillna("") + " " + X_test_nb[VENDOR_COL].fillna("")
)

# 2) Vectorizer: bag-of-words with limited vocab
bow_vec = CountVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

nb_clf = MultinomialNB()

pipe_nb = Pipeline(
    steps=[
        ("bow", bow_vec),
        ("clf", nb_clf),
    ]
)

# 3) Train on text only
pipe_nb.fit(X_train_nb["text_combo"], y_train)

# 4) Evaluate
y_pred_nb = pipe_nb.predict(X_test_nb["text_combo"])

print("=== Multinomial Naive Bayes (BOW) ===")
print(classification_report(y_test, y_pred_nb))


=== Multinomial Naive Bayes (BOW) ===
              precision    recall  f1-score   support

           0       0.93      1.00      0.96        26
           1       0.88      0.86      0.87       209
           2       0.75      0.84      0.79        55
           3       0.94      0.91      0.93       128
           4       0.97      0.95      0.96       151
           5       0.97      0.98      0.97       115
           6       0.76      0.71      0.73        76
           7       0.91      0.93      0.92        90
           8       0.88      0.91      0.89       125

    accuracy                           0.90       975
   macro avg       0.89      0.90      0.89       975
weighted avg       0.90      0.90      0.90       975



Sentence embeddings + XGBoost

In [9]:


from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler

# 1) Prepare text inputs (combine description + vendor)
train_texts = (
    X_train[TEXT_COL].fillna("") + " " + X_train[VENDOR_COL].fillna("")
).tolist()
test_texts = (
    X_test[TEXT_COL].fillna("") + " " + X_test[VENDOR_COL].fillna("")
).tolist()

# 2) Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# 3) Compute embeddings
X_train_emb = embed_model.encode(train_texts, batch_size=64, show_progress_bar=True)
X_test_emb = embed_model.encode(test_texts, batch_size=64, show_progress_bar=True)

# 4) Append numeric features
X_train_num = X_train[NUM_COLS].values
X_test_num = X_test[NUM_COLS].values

# Concatenate: [embedding | numeric]
X_train_emb_full = np.hstack([X_train_emb, X_train_num])
X_test_emb_full = np.hstack([X_test_emb, X_test_num])

# 5) Optionally scale numeric dims (embeddings are roughly normalized already, so not essential)
# scaler = StandardScaler()
# X_train_emb_full[:, -len(NUM_COLS):] = scaler.fit_transform(X_train_emb_full[:, -len(NUM_COLS):])
# X_test_emb_full[:, -len(NUM_COLS):] = scaler.transform(X_test_emb_full[:, -len(NUM_COLS):])

# 6) XGBoost classifier on embeddings
xgb_emb_clf = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
)

xgb_emb_clf.fit(X_train_emb_full, y_train)
y_pred_emb = xgb_emb_clf.predict(X_test_emb_full)

print("=== Sentence embeddings + XGBoost ===")
print(classification_report(y_test, y_pred_emb))


C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 16/16 [00:02<00:00,  6.66it/s]


=== Sentence embeddings + XGBoost ===
              precision    recall  f1-score   support

           0       1.00      0.96      0.98        26
           1       0.88      0.91      0.89       209
           2       0.78      0.84      0.81        55
           3       0.95      0.95      0.95       128
           4       0.99      1.00      1.00       151
           5       0.97      0.98      0.98       115
           6       0.84      0.68      0.75        76
           7       0.94      0.93      0.94        90
           8       0.89      0.90      0.90       125

    accuracy                           0.92       975
   macro avg       0.92      0.91      0.91       975
weighted avg       0.92      0.92      0.92       975

